# Email Spam Classifier

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_curve, auc, ConfusionMatrixDisplay, RocCurveDisplay

sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Load Dataset

In [ ]:
spambase = fetch_ucirepo(id=94)
X = spambase.data.features
y = spambase.data.targets

print(f"Features shape: {X.shape}")
print(f"Targets shape: {y.shape}")
print("\nMissing values in X:", X.isnull().sum().sum())
print("Missing values in y:", y.isnull().sum().sum())

## 2. Preprocessing

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
y_train = y_train.values.ravel()
y_test = y_test.values.ravel()

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Data split and scaled successfully.")

## 3. Model Training and Evaluation

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "SVM": SVC(probability=True),
    "k-NN": KNeighborsClassifier(),
    "Naive Bayes": GaussianNB()
}

results = {}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    results[name] = {
        "accuracy": acc,
        "cm": confusion_matrix(y_test, y_pred),
        "y_proba": y_proba
    }
    
    print(f"{name} Accuracy: {acc:.4f}")

## 4. Comparison of Algorithms

In [ ]:
acc_df = pd.DataFrame({
    "Model": results.keys(),
    "Accuracy": [r["accuracy"] for r in results.values()]
})

plt.figure(figsize=(10, 6))
sns.barplot(x="Model", y="Accuracy", data=acc_df, hue="Model", palette="viridis", legend=False)
plt.ylim(0.8, 1.0)
plt.title("Model Accuracy Comparison")
plt.show()

## 5. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
axes = axes.ravel()

for i, (name, data) in enumerate(results.items()):
    disp = ConfusionMatrixDisplay(confusion_matrix=data["cm"], display_labels=['Ham', 'Spam'])
    disp.plot(ax=axes[i], cmap='Blues', colorbar=False)
    axes[i].set_title(f"{name} Confusion Matrix")

plt.tight_layout()
plt.show()

## 6. ROC Curves

In [ ]:
plt.figure(figsize=(10, 8))

for name, data in results.items():
    fpr, tpr, _ = roc_curve(y_test, data["y_proba"])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {roc_auc:.2f})')

plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves Comparison')
plt.legend(loc="lower right")
plt.grid(True)
plt.show()